In [8]:
from transformers import MusicgenForConditionalGeneration, AutoProcessor
import torch

processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small")

inputs = processor(
    text=["Jesus Christ"],
    return_tensors="pt"
)

audio_values = model.generate(**inputs, max_new_tokens=256)


print(audio_values.shape)



torch.Size([1, 1, 161920])


In [9]:
import numpy as np
from scipy.io import wavfile

# Remove batch dimension
audio = audio_values[0]

# Remove channel dimension if mono
if audio.dim() == 2:
    audio = audio.squeeze(0)

# Convert to numpy
audio = audio.cpu().numpy()

# Normalize safely to int16
audio = audio / np.max(np.abs(audio))
audio = (audio * 32767).astype(np.int16)

# Use correct sample rate from model config
sample_rate = model.config.audio_encoder.sampling_rate

wavfile.write("generated_music_jesus.wav", sample_rate, audio)